In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import pandas as pd
import joblib

app = Flask(__name__)
CORS(app) 


model = joblib.load('resume_screening_model.pkl')
edu_encoder = joblib.load('edu_encoder.pkl')
target_encoder = joblib.load('target_encoder.pkl')


@app.route('/predict', methods=['POST'])
def predict():
    try:
        
        data = request.json
        
        
        candidate_data = pd.DataFrame([data])
        
        candidate_data['education_level'] = edu_encoder.transform(candidate_data['education_level'])
        
        prediction = model.predict(candidate_data)
        
        final_decision = target_encoder.inverse_transform(prediction)[0]
        
        return jsonify({'decision': final_decision})
    
    except Exception as e:
        return jsonify({'error': str(e)})

if __name__ == '__main__':
    print("🚀 AI Server is running and listening for resumes...")
    app.run(debug=True, port=5000)